# Cell 1 — Installs & headless display


In [ ]:

import subprocess, sys, os, time, platform

os.environ["SDL_VIDEODRIVER"]          = "offscreen"
os.environ["SDL_AUDIODRIVER"]          = "disk"
os.environ["DISPLAY"]                  = ":1"
os.environ["LIBGL_ALWAYS_SOFTWARE"]    = "1"
os.environ["MESA_GL_VERSION_OVERRIDE"] = "3.3"
os.environ["GALLIUM_DRIVER"]           = "llvmpipe"
print("Headless env vars set.")     

def pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

pip("vizdoom", "opencv-python-headless", "tensorboard")

if platform.system() == "Linux":
    try:
        subprocess.run(
            "which Xvfb >/dev/null 2>&1 || (apt-get update -qq && apt-get install -y -q xvfb)",
            shell=True, check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        subprocess.Popen(
            ["Xvfb", ":1", "-screen", "0", "1024x768x24", "-ac",
             "+extension", "GLX", "+render", "-noreset"],
            stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
        )
        time.sleep(2)
        print("Xvfb virtual display started on :1")
    except Exception as e:
        print(f"     unavailable (SDL offscreen fallback active): {e}")

import math, random, copy
from dataclasses import dataclass, field
from collections import namedtuple
from typing import List

import numpy as np
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.tensorboard import SummaryWriter

import vizdoom as vzd
from vizdoom import DoomGame , GameVariable, Button, ScreenResolution, ScreenFormat, Mode 

print(f"VizDoom : {vzd.__version__}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device  : {DEVICE}")

# Cell 2 — Config


In [ ]:
@dataclass
class Config:
    # Screen
    height      : int   = 60
    width       : int   = 108
    gray        : bool  = False
    frame_skip  : int   = 4

    # Network
    hidden_dim   : int   = 512
    n_rec_layers : int   = 1
    dropout      : float = 0.0
    dueling      : bool  = True

    # Training
    hist_size          : int   = 4
    n_rec_updates      : int   = 4
    batch_size         : int   = 32
    replay_memory_size : int   = 100_000
    learning_rate      : float = 2e-5
    gamma              : float = 0.99
    update_frequency   : int   = 4
    #  was 10_000 training steps = 40k env steps — far too infrequent,
    # caused systematic Q overestimation. 1_250 training steps = 5k env steps.
    target_update_freq : int   = 1_250
    #  was 10.0 — with SmoothL1 loss already bounding gradients, 10.0
    # means effectively no clipping. 1.0 gives real stabilization.
    grad_clip          : float = 1.0
    epsilon_start      : float = 1.0
    epsilon_end        : float = 0.1
    epsilon_steps      : int   = 500_000

    game_variables : list = field(default_factory=lambda: [
        ("health", 201)
    ])

    game_features_str : str = "target,enemy"

    reward_values : dict = field(default_factory=lambda: {
        "BASE_REWARD":  0.0,
        # FIX: was 0.001 — too small to break camping attractor.
        # 0.05 means ~4 units of movement reward per step at normal speed,
        # enough to compete with the incentive to stand still.
        "DISTANCE":     0.005, 
        "KILL":         30.0, 
        "DEATH":       -10.0,
        "SUICIDE":     -10.0,
        "MEDIKIT":      1.0,
        "ARMOR":        1.0,
        "INJURED":     -1.0,
        "WEAPON":       1.0,
        "AMMO":         1.0,
        "USE_AMMO":    -0.2, 
        # penalize standing still — if distance < 5 units apply -0.02.
        # This creates an active anti-camping push on top of the positive reward.
        "STALE":       -0.1,
    })

    action_combinations : str = "turn_lr+move_fb;attack;move_lr"

    # Evaluation / logging
    eval_freq       : int  = 20_000
    eval_time       : int  = 300
    log_frequency   : int  = 50
    dump_freq       : int  = 50_000
    total_steps     : int  = 1_000_000
    n_bots          : int  = 4
    n_eval_episodes : int  = 10
    render          : bool = False

    # add explicit training episode time limit.
    # 2100 game ticks = 60 seconds at 35fps. Without this, episodes only end
    # on is_episode_finished() which can run for hours in deathmatch.
    train_episode_time : int = 2100

    # Scenario
    scenario       : str  = "deathmatch"
    # FIX: was [1] only — agent memorized map 1 geometry and camped.
    # Multiple maps force generalizable behavior.
    map_ids_train  : list = field(default_factory=lambda: [1, 2, 3, 4, 5])
    map_ids_test   : list = field(default_factory=lambda: [1, 2, 3, 4, 5])

    # Derived
    n_fm         : int  = 3
    n_features   : int  = 0
    n_variables  : int  = 0
    n_actions    : int  = 0
    game_features: list = field(default_factory=list)

    def __post_init__(self):
        self.n_fm = 1 if self.gray else 3
        feats = ["target", "enemy", "health", "weapon", "ammo"]
        split = [x for x in self.game_features_str.split(",") if x]
        self.game_features = [x in split for x in feats]
        self.n_features = len(split)
        self.n_variables = len(self.game_variables)


CFG = Config()
print(f"Config: {CFG.height}x{CFG.width}, gray={CFG.gray}, "
      f"hist={CFG.hist_size}, n_rec_updates={CFG.n_rec_updates}")
print(f"  features={CFG.n_features} ({CFG.game_features_str})")
print(f"  variables={CFG.n_variables} {[v[0] for v in CFG.game_variables]}")
print(f"  n_eval_episodes={CFG.n_eval_episodes}")
print(f"  train_episode_time={CFG.train_episode_time} ticks")
print(f"  map_ids_train={CFG.map_ids_train}")
print(f"  target_update_freq={CFG.target_update_freq} train steps = "
      f"{CFG.target_update_freq * CFG.update_frequency} env steps")

# Cell 3 — Action space

In [ ]:

ACTION_CATEGORIES = {
    "turn_lr": ["TURN_LEFT",  "TURN_RIGHT"],
    "look_ud": ["LOOK_UP",    "LOOK_DOWN"],
    "move_fb": ["MOVE_FORWARD","MOVE_BACKWARD"],
    "move_lr": ["MOVE_LEFT",  "MOVE_RIGHT"],
    "attack":  ["ATTACK"],
}
# create a combination of actions with cartesian product turn_lr+move_fb;attack;move_lr this string defines all the actions 

def create_action_set(action_combinations_str):
    action_subsets = []
    # we split the branches of actions , move or turn or shoot 
    for group_str in action_combinations_str.split(";"):
        buttons = []
        # here we split each branch to the specific action
        for cat in group_str.split("+"):
            buttons.extend(ACTION_CATEGORIES[cat])
            # we add a none that defines not taking any action in each branch 
        action_subsets.append([None] + buttons)

    # here is the actual cartesian product 
    action_set = [[]]
    for subset in action_subsets:
        action_set = [prev + [x] for prev in action_set for x in subset]

    action_set = [[y for y in x if y is not None] for x in action_set]
    action_set = [z for z in action_set if len(z) > 0]
    return action_set


_BUTTON_NAMES = [
    "MOVE_FORWARD", "MOVE_BACKWARD",
    "TURN_LEFT",    "TURN_RIGHT",
    "MOVE_LEFT",    "MOVE_RIGHT",
    "ATTACK",
    "SELECT_WEAPON2", "SELECT_WEAPON3", "SELECT_WEAPON4",
    "SELECT_WEAPON5", "SELECT_WEAPON6", "SELECT_WEAPON7",
]

_BUTTON_ENUMS = [
    Button.MOVE_FORWARD, Button.MOVE_BACKWARD,
    Button.TURN_LEFT,    Button.TURN_RIGHT,
    Button.MOVE_LEFT,    Button.MOVE_RIGHT,
    Button.ATTACK,
    Button.SELECT_WEAPON2, Button.SELECT_WEAPON3, Button.SELECT_WEAPON4,
    Button.SELECT_WEAPON5, Button.SELECT_WEAPON6, Button.SELECT_WEAPON7,
]


class ActionBuilder:
    def __init__(self, cfg):
        self.cfg = cfg
        self.mapping = {btn: i for i, btn in enumerate(_BUTTON_NAMES)}
        # create combinations 
        action_list  = create_action_set(cfg.action_combinations)
        self.doom_actions = []
        # we generate all possible action combination in a form of 13 binary vector
        for action in action_list:
            vec = [False] * len(_BUTTON_NAMES)
            for btn in action:
                if btn in self.mapping:
                    vec[self.mapping[btn]] = True
            self.doom_actions.append(vec)
        cfg.n_actions = len(self.doom_actions)
    # get action vector with an idx 
    def get_action(self, action_id):
        return list(self.doom_actions[action_id])


action_builder = ActionBuilder(CFG)
print(f"Action space: {len(action_builder.doom_actions)} discrete actions")
for i, a in enumerate(action_builder.doom_actions[:5]):
    btns = [b for b, v in zip(_BUTTON_NAMES, a) if v]
    print(f"  [{i}] {btns}")
print(f"  ... ({len(action_builder.doom_actions) - 5} more)")
print(f"Total actions: {CFG.n_actions}")

# Cell 4 — Reward builder


In [ ]:

WEAPONS_PREFERENCES = [
    ("bfg9000",        "cells",   7),
    ("plasmarifle",    "cells",   6),
    ("rocketlauncher", "rockets", 5),
    ("chaingun",       "bullets", 4),
    ("shotgun",        "shells",  3),
    ("pistol",         "bullets", 2),
]


class RewardBuilder:
    def __init__(self, values):
        self.values = dict(values)
        self.reset()

    def reset(self):
        self._reward = self.values["BASE_REWARD"]

    def distance(self, d): self._reward += self.values["DISTANCE"] * d
    def kill(self, n):     self._reward += self.values["KILL"] * n
    def death(self):       self._reward += self.values["DEATH"]
    def suicide(self):     self._reward += self.values["SUICIDE"]
    def medikit(self, hp): self._reward += self.values["MEDIKIT"]
    def armor(self):       self._reward += self.values["ARMOR"]
    def injured(self, hp): self._reward += self.values["INJURED"]
    def weapon(self):      self._reward += self.values["WEAPON"]
    def ammo(self):        self._reward += self.values["AMMO"]
    def use_ammo(self):    self._reward += self.values["USE_AMMO"]
    def stale(self):       self._reward += self.values.get("STALE", 0.0)
    @property
    def reward(self):
        r = self._reward
        self.reset()
        return r


print(f"RewardBuilder ready: {CFG.reward_values}")

# Cell 5 — DoomEnv
   • KILLCOUNT registered as a game variable (kills were silently 0)
   • _compute_reward uses kill_count (KILLCOUNT) not frag_count (FRAGCOUNT)
   • process_screen handles planar CRCGCB (C,H,W) format correctly

In [ ]:
GameState = namedtuple("GameState", ["screen", "variables", "features"])


def process_screen(screen_buffer, cfg):
    if screen_buffer is None:
        return np.zeros((cfg.n_fm, cfg.height, cfg.width), dtype=np.uint8)
    if screen_buffer.ndim == 3 and screen_buffer.shape[0] in (1, 3):
        img = screen_buffer.transpose(1, 2, 0)
    else:
        img = screen_buffer
    img = cv2.resize(img.astype(np.uint8), (cfg.width, cfg.height),
                     interpolation=cv2.INTER_AREA)
    if cfg.gray:
        if img.ndim == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        return img[np.newaxis].astype(np.uint8)
    else:
        if img.ndim == 2:
            img = np.stack([img, img, img], axis=0)
        else:
            img = img.transpose(2, 0, 1)
        return img.astype(np.uint8)


_ENEMY_KEYWORDS = (
    "zombieman", "shotgunguy", "heavyweapondude", "doomimp", "imp",
    "demon", "spectre", "lostsoul", "cacodemon", "baronofhell",
    "hellknight", "revenant", "mancubus", "archvile",
    "spidermastermind", "cyberdemon",
)
_WEAPON_PICKUP_NAMES = frozenset((
    "shotgun", "supershotgun", "chaingun", "rocketlauncher",
    "plasmarifle", "bfg9000", "chainsaw",
))
_HEALTH_KEYWORDS  = ("medikit", "stimpack", "soulsphere", "megasphere", "healthbonus")
_AMMO_KEYWORDS    = ("clip", "ammobox", "shells", "shellbox", "rocket",
                     "energycell", "backpack")

# get features from the game engine 
def detect_game_features(labels_buffer, labels, ga  me_features_flags):
    # defined list of features needed from the game screen/ram
    feature_names = ["target", "enemy", "health", "weapon", "ammo"]
    # only keep the features defined on the config later / in my case only enemy and target needed
    active   = [n for n, flag in zip(feature_names, game_features_flags) if flag]
    # init a dict for all the features needed in active 
    detected = {n: 0 for n in active}
    
    # variables provided by the doom engine to detect object on ram / screen 
    if labels_buffer is not None and labels is not None:
        # we loop on all the items and entities on the screen 
        for lbl in labels:
            # we provided a lowered name to match it with our pre defined list
            name = lbl.object_name.lower() if hasattr(lbl, "object_name") else ""
            # here if we got an enemy or target from our list we activate the features 
            if any(k in name for k in _ENEMY_KEYWORDS):
                if "enemy"  in detected: detected["enemy"]  = 1
                if "target" in detected: detected["target"] = 1
            if any(k in name for k in _HEALTH_KEYWORDS):
                if "health" in detected: detected["health"] = 1
            if name in _WEAPON_PICKUP_NAMES:
                if "weapon" in detected: detected["weapon"] = 1
            if any(k in name for k in _AMMO_KEYWORDS):
                if "ammo"   in detected: detected["ammo"]   = 1
    # return the features on an array like something [1,1,1,1,0] for pytorch to calculate
    return [detected[n] for n in active]


# min distance passed to be considered as a stale movememnt
_STALE_DISTANCE_THRESHOLD = 5.0


class DoomEnv:
    def __init__(self, cfg, action_builder):
        # init enviroment vars , like game , and action builder mapping ,state and next state , rewards ...
        self.cfg             = cfg
        self.action_builder  = action_builder
        self.mapping         = action_builder.mapping
        self.game            = DoomGame()
        self.reward_builder  = RewardBuilder(cfg.reward_values)
        self.properties      = None
        self.prev_properties = None
        self._screen_buffer  = None
        self._labels_buffer  = None
        self._labels         = None
        self._reward         = 0.0
    # function to start the env
    def start(self, map_id=1, episode_time=None, log_events=False):
        # basically define the map and the episode time and the resolution and
        #  game sound/ graphic configs all hidden ofcourse for faster training 

        g = self.game
        g.set_doom_map(f"MAP{map_id:02d}")
        g.set_screen_resolution(ScreenResolution.RES_320X240)
        g.set_screen_format(ScreenFormat.CRCGCB)
        g.set_depth_buffer_enabled(False)
        g.set_labels_buffer_enabled(True)
        g.set_automap_buffer_enabled(False)
        g.set_objects_info_enabled(True)
        g.set_sectors_info_enabled(False)
        g.set_render_hud(False)
        g.set_render_crosshair(False)
        g.set_render_decals(False)
        g.set_render_particles(False)
        g.set_window_visible(self.cfg.render)
        g.set_mode(Mode.PLAYER)
        g.set_living_reward(0)
       
        g.set_doom_skill(4)
        # here just define all the vars needed from the game like health ...etc
        g.set_available_buttons(_BUTTON_ENUMS)

        g.add_available_game_variable(GameVariable.HEALTH)
        g.add_available_game_variable(GameVariable.ARMOR)
        g.add_available_game_variable(GameVariable.AMMO2)
        g.add_available_game_variable(GameVariable.AMMO3)
        g.add_available_game_variable(GameVariable.AMMO5)
        g.add_available_game_variable(GameVariable.AMMO6)
        g.add_available_game_variable(GameVariable.WEAPON2)
        g.add_available_game_variable(GameVariable.WEAPON3)
        g.add_available_game_variable(GameVariable.WEAPON4)
        g.add_available_game_variable(GameVariable.WEAPON5)
        g.add_available_game_variable(GameVariable.WEAPON6)
        g.add_available_game_variable(GameVariable.WEAPON7)
        g.add_available_game_variable(GameVariable.SELECTED_WEAPON)
        g.add_available_game_variable(GameVariable.POSITION_X)
        g.add_available_game_variable(GameVariable.POSITION_Y)
        g.add_available_game_variable(GameVariable.FRAGCOUNT)
        g.add_available_game_variable(GameVariable.KILLCOUNT)
        g.add_available_game_variable(GameVariable.DEATHCOUNT)

        # episode timout 
        if episode_time is not None:
            g.set_episode_timeout(int(episode_time))

        # we init the game and the episode
        g.init()
        g.new_episode()

        # add bots and monsters 

        for _ in range(self.cfg.n_bots):
            g.send_game_command("addbot")

        # get state 
        state = g.get_state()
        if state is not None:
            self._screen_buffer = state.screen_buffer
            self._labels_buffer = state.labels_buffer
            self._labels        = state.labels
        
        self.properties      = self._get_properties()
        self.prev_properties = dict(self.properties)
        self.reward_builder.reset()
        self.properties["death_count"]      = 0.0
        self.prev_properties["death_count"] = 0.0
        print(f"Game started on map {map_id}")

    # helper function to get the game vars / features 
    def _gv(self, var):
        try:
            return float(self.game.get_game_variable(var))
        except Exception:
            return 0.0
    # return all the features needed for the game , like ammo ,health kills basically stuff that the player usually see on the screen 
    def _get_properties(self):
        gv = self._gv
        return {
            "health":         max(0.0, gv(GameVariable.HEALTH)),
            "armor":          max(0.0, gv(GameVariable.ARMOR)),
            "bullets":        max(0.0, gv(GameVariable.AMMO2)),
            "shells":         max(0.0, gv(GameVariable.AMMO3)),
            "rockets":        max(0.0, gv(GameVariable.AMMO5)),
            "cells":          max(0.0, gv(GameVariable.AMMO6)),
            "pistol":         int(gv(GameVariable.WEAPON2) > 0),
            "shotgun":        int(gv(GameVariable.WEAPON3) > 0),
            "chaingun":       int(gv(GameVariable.WEAPON4) > 0),
            "rocketlauncher": int(gv(GameVariable.WEAPON5) > 0),
            "plasmarifle":    int(gv(GameVariable.WEAPON6) > 0),
            "bfg9000":        int(gv(GameVariable.WEAPON7) > 0),
            "sel_weapon":     int(gv(GameVariable.SELECTED_WEAPON)),
            "pos_x":          gv(GameVariable.POSITION_X),
            "pos_y":          gv(GameVariable.POSITION_Y),
            "frag_count":     gv(GameVariable.FRAGCOUNT),
            "kill_count":     gv(GameVariable.KILLCOUNT),
            "death_count":    gv(GameVariable.DEATHCOUNT),
        }

    # here we calc the reward 
    def _compute_reward(self):
        if self.properties is None or self.prev_properties is None:
            return
        # update the St and St+1
        p  = self.properties
        pp = self.prev_properties
        rb = self.reward_builder
        rb.reset()
        
        # calc the reward from distance passed
        dx   = p["pos_x"] - pp["pos_x"]
        dy   = p["pos_y"] - pp["pos_y"]
        dist = math.sqrt(dx * dx + dy * dy)
        if dist > 0:
            rb.distance(dist)

        # penalty for stale
        if dist < _STALE_DISTANCE_THRESHOLD:
            rb.stale()
        # current kills in this step 
        dk = p["kill_count"] - pp["kill_count"]
        if dk > 0:
            rb.kill(dk)
        # how many times we got killed on this step 
        dd = p["death_count"] - pp["death_count"]
        if dd > 0:
            rb.death()
        # usually same as kills but also counting suicide so if you kill ur self  we get -1 added , the frag count = killcount - suicide
        if p["frag_count"] < pp["frag_count"]:
            rb.suicide()

        # health change
        dh = p["health"] - pp["health"]
        if   dh > 0: rb.medikit(dh)
        elif dh < 0: rb.injured(dh)

        if p["armor"] - pp["armor"] > 0:
            rb.armor()
        # weapons founded 
        for wp in ["pistol", "shotgun", "chaingun",
                   "rocketlauncher", "plasmarifle", "bfg9000"]:
            if pp.get(wp, 0) == 0 and p.get(wp, 0) == 1:
                rb.weapon()
        # ammo used 
        for am in ["bullets", "shells", "rockets", "cells"]:
            d = p.get(am, 0) - pp.get(am, 0)
            if   d > 0: rb.ammo()
            elif d < 0: rb.use_ammo()
        # final reward
        self._reward = rb.reward
    
    def observe_state(self, last_states):
        # we get the screen preprocessing and the game features ready 
        screen        = process_screen(self._screen_buffer, self.cfg)
        game_features = detect_game_features(
            self._labels_buffer, self._labels, self.cfg.game_features)
        variables = []
        # we look through the config vars , like health ... and we turn those vars into something acceptable as a state 
        for name, n_vals in self.cfg.game_variables:
            val = self.properties.get(name, 0) if self.properties else 0
            variables.append(max(0, min(n_vals - 1, int(val))))
        # append the total state vars
        last_states.append(GameState(screen, variables, game_features))
        # here if we got only one frame we just dup it to be 4 so it can fit our cnn lstm architecture 
        # else we append the new frame of stats and delete the last one

        if len(last_states) == 1:
            last_states.extend([last_states[0]] * (self.cfg.hist_size - 1))
        else:
            assert len(last_states) == self.cfg.hist_size + 1
            del last_states[0]
        return screen, game_features


    def make_action(self, action_id):
        # we get the action vector we need 
        action = self.action_builder.get_action(action_id)
        # here we force the best weapon choice on the game 
        # we loop over the whole weopons array and we choose the stronger and the one with enough ammo 
         
        if self.properties:
            for wp_name, wp_ammo, wp_id in WEAPONS_PREFERENCES:
                min_ammo = 40 if wp_name == "bfg9000" else 1
                if (self.properties.get(wp_name, 0) > 0 and
                        self.properties.get(wp_ammo, 0) >= min_ammo):
                    if self.properties.get("sel_weapon", 1) != wp_id:
                        sw_key = f"SELECT_WEAPON{wp_id}"
                        if sw_key in self.mapping:
                            idx = self.mapping[sw_key]
                            while len(action) <= idx:
                                action.append(False)
                            action[idx] = True
                    break
        # engine execute the action and get the state after 4 frames 
        self.game.make_action(action, self.cfg.frame_skip)
        state = self.game.get_state()
        if state is not None:
            self._screen_buffer = state.screen_buffer
            self._labels_buffer = state.labels_buffer
            self._labels        = state.labels
        else:
            self._screen_buffer = None
            self._labels_buffer = None
            self._labels        = None

        # update state St and St+1 and calc reward
        self.prev_properties = dict(self.properties) if self.properties else {}
        self.properties      = self._get_properties()
        self._compute_reward()
        return self._reward

    def is_final(self):
       
        return self.game.is_episode_finished()

    def new_episode(self):
        # tell game to make a new episode
        self.game.new_episode()
        
        # remove the old bots via a cmd

        self.game.send_game_command("removebots")  
        # add the bots again 

        for _ in range(self.cfg.n_bots):
            self.game.send_game_command("addbot")
        # get inital state and reset all the vars  and the buffers and the semantics of the game from the c++ engine 
        state = self.game.get_state()
        if state is not None:
            self._screen_buffer = state.screen_buffer
            self._labels_buffer = state.labels_buffer
            self._labels        = state.labels
        self.properties      = self._get_properties()
        self.prev_properties = dict(self.properties)
        self.reward_builder.reset()
        self.properties["death_count"]      = 0.0
        self.prev_properties["death_count"] = 0.0
        self.properties["kill_count"]       = 0.0
        self.prev_properties["kill_count"]  = 0.0
        self.properties["frag_count"]       = 0.0
        self.prev_properties["frag_count"]  = 0.0
    def close(self):
        try:
            self.game.close()
        except Exception:
            pass
    

print("DoomEnv ready")

# Cell 6 — Replay Memory
  • Cursor check covers the entire sampled window, not just one end
  • Modulo wrap applied before indexing into circular arrays

In [ ]:

class ReplayMemory:
    def __init__(self, max_size, screen_shape, n_variables, n_features):
        assert len(screen_shape) == 3
        # vars and state of the replay memory 
        self.max_size     = max_size
        self.screen_shape = screen_shape
        self.n_variables  = n_variables
        self.n_features   = n_features 
        self.cursor = 0
        self.full   = False

        self.screens   = np.zeros((max_size,) + screen_shape, dtype=np.uint8)
        self.variables = (np.zeros((max_size, n_variables), dtype=np.int32)
                          if n_variables else None)
        self.features  = (np.zeros((max_size, n_features),  dtype=np.int32)
                          if n_features  else None)
        self.actions   = np.zeros(max_size, dtype=np.int32)
        self.rewards   = np.zeros(max_size, dtype=np.float32)
        self.isfinal   = np.zeros(max_size, dtype=np.bool_)
        # calculate the size of memory needed for the img
        mem_gb = (max_size * int(np.prod(screen_shape))) / (1024 ** 3)
        print(f"ReplayMemory: {max_size:,} slots, screen={screen_shape}, "
              f"vars={n_variables}, feats={n_features}")
        print(f"  Screen memory: {mem_gb:.2f} GB")

    @property
    def size(self):
        return self.max_size if self.full else self.cursor
    # add state to the buffer 
    def add(self, screen, variables, features, action, reward, is_final):
        self.screens[self.cursor] = screen
        if self.n_variables and variables is not None:
            self.variables[self.cursor] = variables
        if self.n_features and features is not None:
            self.features[self.cursor]  = features
        self.actions[self.cursor] = action
        self.rewards[self.cursor] = reward
        self.isfinal[self.cursor] = is_final 
        self.cursor += 1
        if self.cursor >= self.max_size:
            self.cursor = 0
            self.full   = True
     # return batch for training 
    def get_batch(self, batch_size, hist_size):

        seq_len  = hist_size + 1 
        indices  = []
        attempts = 0
        max_attempts = batch_size * 20 
    
        while len(indices) < batch_size and attempts < max_attempts:
            attempts += 1
           # if we dont have enough frames to begin with , break no training 
            if (self.max_size if self.full else self.cursor) < seq_len:
                break
            # find a random number and get the starting and the ending idx
            end_idx   = random.randint(seq_len - 1, (self.max_size if self.full else self.cursor) - 1)
            start_idx = end_idx - seq_len + 1 
    
            # turn the idx to circular to prevent out of the buffer index error 
            real_indices = [(start_idx + i) %  self.max_size for i in range(seq_len)] 
            boundary_crossed = False
            # check if there is a state where the frame is final and we break from it because it will kill the lstm learning path 
            for ri in real_indices[:-1]:   # check all except the last
                if self.isfinal [ri]:
                    boundary_crossed = True
                    break 
            if boundary_crossed:
                continue
    
            indices.append(real_indices)
        
        # if we cant find a good sequence for the lstm we ignore that rule and just look randomly
        if len(indices) == 0:
            
            indices = []
            for _ in range(batch_size):
                end_idx   = random.randint(seq_len - 1, (self.max_size if self.full else self.cursor) - 1)
                start_idx = end_idx - seq_len + 1
                indices.append([(start_idx + i) % self.max_size 
                                for i in range(seq_len)])
    
        # Build batch arrays from sampled indices
        B = len(indices)
        screens   = np.stack([self.screens[idx]   for idx in indices], axis=0)
        variables = np.stack([self.variables[idx] for idx in indices], axis=0) \
                    if self.n_variables > 0 else None
        features  = np.stack([self.features[idx]  for idx in indices], axis=0) \
                    if self.n_features  > 0 else None
        actions   = np.array([[self.actions[i]   for i in idx[:-1]] for idx in indices])
        rewards   = np.array([[self.rewards[i]   for i in idx[:-1]] for idx in indices])
        isfinal   = np.array([[self.isfinal[i]  for i in idx[:-1]] for idx in indices])
    
        return {
            "screens":   screens,
            "variables": variables,
            "features":  features,
            "actions":   actions,
            "rewards":   rewards,
            "isfinal":   isfinal,
        }


print("ReplayMemory class ready")

# Cell 7 — Network (DRQN)
   • proj_game_features now reads from LSTM output (hidden_dim) instead
     of directly from CNN output.  The old version bypassed the LSTM
     entirely, so the network had no incentive to learn object permanence
     (remembering enemies that walked behind cover).


In [ ]:

class DRQNModule(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg        = cfg
        self.n_actions  = cfg.n_actions
        self.n_features = cfg.n_features
        self.n_variables= cfg.n_variables
        self.dropout_p  = cfg.dropout

        # CNN 
        self.conv = nn.Sequential(
            nn.Conv2d(cfg.n_fm, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32,       64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64,       64, kernel_size=3, stride=1), nn.ReLU(),
        )
        # auto scaler for variable size of images it takes a 0 value images and get stride and padding 
        with torch.no_grad():
            dummy = torch.zeros(1, cfg.n_fm, cfg.height, cfg.width)
            self.conv_output_dim = int(np.prod(self.conv(dummy).shape[1:]))

        #  Game variable embeddings to learn a vector embeddings for health ammo 
        self.game_variable_embeddings = nn.ModuleList()
        total_emb_dim = 0
        for _, n_vals in cfg.game_variables:
            emb_dim = max(4, min(32, n_vals // 4))
            self.game_variable_embeddings.append(nn.Embedding(n_vals, emb_dim))
            total_emb_dim += emb_dim

        # calculate the total size flowing from the cnn and the game var embedding vector to be an input for the lstm 
        self.rnn_input_dim = self.conv_output_dim + total_emb_dim

        if self.dropout_p > 0:
            self.dropout_layer = nn.Dropout(p=self.dropout_p)
        # we pass the input to the lstm which compresses it to the hidden size vector hidden_size 
        # LSTM
        self.rnn = nn.LSTM(
            input_size  = self.rnn_input_dim,
            hidden_size = cfg.hidden_dim,
            num_layers  = cfg.n_rec_layers,
            batch_first = True,
        )

        # the Q head which gets the compressed vector from the lstm 
        # and passes it through the q learning network which is split between the action and value heads
        self.proj_action_scores = nn.Linear(cfg.hidden_dim, self.n_actions)
        if cfg.dueling:
            self.proj_state_values = nn.Linear(cfg.hidden_dim, 1)

        # game-feature head reads from hidden_dim (post-LSTM).
        # Was nn.Linear(conv_output_dim, n_features) which skipped LSTM
        # entirely — the gradient could not reinforce temporal memory.
        # learn the game features to force lstm to learn how to focus on learning the past game variables example enemies hiding in a wall 
        if self.n_features > 0:
            self.proj_game_features = nn.Linear(cfg.hidden_dim, self.n_features)

        print(f"DRQNModule: conv_out={self.conv_output_dim}, "
              f"rnn_in={self.rnn_input_dim}, hidden={cfg.hidden_dim}, "
              f"actions={self.n_actions}, features={self.n_features}")

    def _cnn_embed(self, x_screens, x_variables):
        """
        x_screens   : (N, C, H, W)  float
        x_variables : list of n_var tensors (N,) long
        Returns     : (N, rnn_input_dim)
        """
        # flatten the batch of images after normalization 
        N   = x_screens.size(0)
        out = self.conv(x_screens / 255.0).view(N, -1)
        # turns the game variables into an embeddings vector and then concatenate them with the flattened image
        if self.n_variables > 0:
            embs = [self.game_variable_embeddings[i](x_variables[i])
                    for i in range(self.n_variables)]
            out  = torch.cat([out] + embs, dim=1)

        if self.dropout_p > 0:
            out = self.dropout_layer(out)

        return out

    def _q_head(self, h):
        """h : (N, hidden_dim)  →  (N, n_actions)"""
        # runs the output of the lstm h into each head of action and value network and returns the Q value either by duel head or without it 
        if self.cfg.dueling:
            a = self.proj_action_scores(h)
            v = self.proj_state_values(h)
            return v + (a - a.mean(dim=-1, keepdim=True))
        return self.proj_action_scores(h)

    def forward(self, x_screens, x_variables, prev_state):
        # flatten_parameters() must be inside forward(), not __init__.
        # NOTE: do NOT call self.module.eval() / self.module.train() during
        # inference — those mode switches defragment LSTM weights and undo
        # this call. Use torch.no_grad() only for inference instead.
        self.rnn.flatten_parameters()
        # getting the size of the batch and the number of frames 
        B, T = x_screens.size(0), x_screens.size(1)
        
        # sending the cnn a B * T vectors to process 
        cnn_out = self._cnn_embed(
            x_screens.reshape(B * T, *x_screens.shape[2:]),
            [v.reshape(B * T) for v in x_variables],
        )
        # reshape the output of the cnn to get the time indexing that the lstm needs 
        rnn_out, next_state = self.rnn(
            cnn_out.view(B, T, self.rnn_input_dim), prev_state
        )
        rnn_out = rnn_out.contiguous()
        # flatten the output to be a distinct B* T batch of vectors to feed the Q head 
        flat   = rnn_out.reshape(B * T, self.cfg.hidden_dim)
        q_vals = self._q_head(flat).view(B, T, self.n_actions)

        # game feature head forward it reshapes the data for temporal importance again 
        gf_out = None
        if self.n_features > 0:
            gf_out = self.proj_game_features(flat).view(B, T, self.n_features)

        # return the q value the next state and game features 
        return q_vals, gf_out, next_state


net = DRQNModule(CFG).to(DEVICE)
print(f"Total parameters: {sum(p.numel() for p in net.parameters()):,}")

# Cell 8 — Agent

   • Double DQN target: online net picks action, frozen target net scores it
   • GF loss weighted by LAMBDA_GF=0.1 so it can't overpower the TD signal
   • GF loss aligned to the same n_rec_updates timesteps as TD loss
   • LSTM hidden state persists across steps within an episode during
     inference (was re-initialized to zeros on every select_action call)

In [ ]:


LAMBDA_GF = 0.1


class DRQNAgent:
    def __init__(self, cfg, module):
        self.cfg    = cfg
        self.device = DEVICE
        self.module = module
        self.target_module = copy.deepcopy(module)
        #permanently locking the Target Network into Evaluation Mode.
        self.target_module.eval() 

        # define losses and optimizer 
        self.optimizer  = torch.optim.Adam(module.parameters(), lr=cfg.learning_rate)
        self.loss_fn_sc = nn.SmoothL1Loss()
        self.loss_fn_gf = nn.BCEWithLogitsLoss()

        self.replay_memory = ReplayMemory(
            cfg.replay_memory_size,
            (cfg.n_fm, cfg.height, cfg.width),
            cfg.n_variables,
            cfg.n_features,
        )
        
        #when training since we need a previous state we just feed the zero vec , for all the batches at the beginning and use what the lstm outputs as the next thing to put into the lstm next 
        # , so the difference is just the way we start things 
        # , in the live , we also use the same concept but we reuse the handc instead of init_state 


        # defines the immediate hidden state memory h0 and the long term memory c0
        # Fixed zero state used during training (batch_size, not 1)
        h0 = torch.zeros(cfg.n_rec_layers, cfg.batch_size, cfg.hidden_dim).to(self.device)
        c0 = torch.zeros(cfg.n_rec_layers, cfg.batch_size, cfg.hidden_dim).to(self.device)
        self.init_state_t = (h0, c0)

        self._train_step_count = 0

        # define the update state but init it to 0 in training we need at first the init state 
        # then we will input the hidden state and it will continue the frame looping 
        self.hidden_state = None 
        self._reset_hidden() 

    
    def _reset_hidden(self):
        """Allocate a fresh (1-sample) hidden state for inference."""
        h = torch.zeros(self.cfg.n_rec_layers, 1, self.cfg.hidden_dim).to(self.device)
        c = torch.zeros(self.cfg.n_rec_layers, 1, self.cfg.hidden_dim).to(self.device)
        self.hidden_state = (h, c)

    def reset(self):
        """Call at the start of every episode."""
        self._reset_hidden()

    def _epsilon(self, n_iter):
        cfg = self.cfg
        if n_iter >= cfg.epsilon_steps:
            return cfg.epsilon_end
        t = n_iter / cfg.epsilon_steps
        return cfg.epsilon_start + t * (cfg.epsilon_end - cfg.epsilon_start) 

    def select_action(self, last_states, n_iter):
        # we update the epsilon and then we run the input throught the network to get out next action 
        eps = self._epsilon(n_iter)
    
        with torch.no_grad():
            s = last_states[-1]

            screen = torch.FloatTensor(
                s.screen.astype(np.float32)
            ).unsqueeze(0).unsqueeze(0).to(self.device)
    
            var_list = (
                [torch.LongTensor([[s.variables[i]]]).to(self.device)
                 for i in range(self.cfg.n_variables)]
                if self.cfg.n_variables else []
            )
    
            q_vals, gf_out, self.hidden_state = self.module(
                screen, var_list, self.hidden_state
            )
    
            if random.random() < eps:
                action = random.randint(0, self.cfg.n_actions - 1)
            else:
                action = int(q_vals[0, 0].argmax().item())
    
            pred_features = (gf_out[0, 0].sigmoid().cpu().numpy() 
                             if gf_out is not None else None)
    
        return action, pred_features, eps

    # store the state transition into the replay memory
    def store_transition(self, last_states, action, reward, is_final):
        reward = float(np.clip(reward, -5.0, 5.0))
        s = last_states[-1]
        self.replay_memory.add(
            screen   = s.screen,
            variables= s.variables,
            features = s.features,
            action   = action,
            reward   = reward,
            is_final = is_final,
        )

    def train_step(self, n_iter):
        if n_iter % self.cfg.update_frequency != 0:
            return None
        if self.replay_memory.size < self.cfg.batch_size:
            return None

        # hist_size for get_batch = burn-in frames + gradient frames - 1
        hist_size = self.cfg.hist_size + self.cfg.n_rec_updates - 1
        seq_len   = hist_size + 1   # number of screen frames in each sample
        memory    = self.replay_memory.get_batch(self.cfg.batch_size, hist_size)

        screens  = torch.FloatTensor(
            memory["screens"].astype(np.float32)).to(self.device)          # (B, seq_len, C, H, W)
        variables = (
            torch.LongTensor(memory["variables"].astype(np.float32)).to(self.device)
            if self.cfg.n_variables else None)
        features = (
            torch.LongTensor(memory["features"].astype(np.float32)).to(self.device)
            if self.cfg.n_features else None)
        actions  = memory["actions"]                                        # (B, hist_size)
        rewards  = torch.FloatTensor(
            memory["rewards"].astype(np.float32)).to(self.device)          # (B, hist_size)
        isfinal  = torch.FloatTensor(
            memory["isfinal"].astype(np.float32)).to(self.device)          # (B, hist_size)

        var_list = ([variables[:, :, i] for i in range(self.cfg.n_variables)]
                    if self.cfg.n_variables else [])

        # ── Online forward pass  
        output_sc, output_gf, _ = self.module(screens, var_list, self.init_state_t)
        # output_sc : (B, seq_len, n_actions)

        # Gather Q(s_t, a_t) for all t in 0..hist_size-1
        n_steps = seq_len - 1   # = hist_size
        mask = torch.zeros(output_sc.size(), dtype=torch.bool, device=self.device)
        for i in range(self.cfg.batch_size):
            for j in range(n_steps):
                mask[i, j, int(actions[i, j])] = True
        scores1 = output_sc.masked_select(mask)   # (B * n_steps,)  

        # ── Double DQN target ─────────────────────────────────────────
        # overestimates.  With 35 actions the bias compounds quickly.
        # Solution: online net selects the greedy action; frozen target
        # net evaluates it.  These two steps use different parameters so
        # the overestimation bias cancels.
        with torch.no_grad():
            # stable estimate of the game 
            tgt_sc, _, _ = self.target_module(screens, var_list, self.init_state_t)
            # online selects: argmax Q_online(s_{t+1})
            online_next_actions = (output_sc[:, 1:, :]
                                   .detach()
                                   .argmax(dim=2, keepdim=True))           # (B, n_steps, 1)
            # target evaluates: Q_target(s_{t+1}, a_online)
            target_q = tgt_sc[:, 1:, :].gather(2, online_next_actions).squeeze(2)
            scores2  = rewards + self.cfg.gamma * target_q * (1 - isfinal)
            # scores2 : (B, n_steps)

        n_u = self.cfg.n_rec_updates

        # ── TD loss — last n_u timesteps only ─────────────────────────
        loss_sc = self.loss_fn_sc(
            scores1.view(self.cfg.batch_size, -1)[:, -n_u:],
            scores2[:, -n_u:],
        )

        # ── GF loss — aligned to same n_u timesteps ───────────────────
        
        loss_gf = torch.tensor(0.0, device=self.device)
        if (self.cfg.n_features > 0
                and output_gf is not None
                and features  is not None):
            loss_gf = self.loss_fn_gf(
                output_gf[:, -n_u:, :],
                features[:, -n_u:, :].float(),
            )

        total_loss = loss_sc + LAMBDA_GF * loss_gf   

        self.optimizer.zero_grad()
        total_loss.backward()
        for p in self.module.parameters():
            if p.grad is not None:
                p.grad.data.clamp_(-self.cfg.grad_clip, self.cfg.grad_clip) 
        self.optimizer.step()
        
        # Periodically sync online → target
        self._train_step_count += 1
        if self._train_step_count % self.cfg.target_update_freq == 0:
            self.target_module.load_state_dict(self.module.state_dict())
            self.target_module.eval()

        return {
            "loss_total": total_loss.item(),
            "loss_td":    loss_sc.item(),
            "loss_gf":    loss_gf.item(),
            "mean_q":     output_sc[:, -n_u:, :].max(dim=2)[0].mean().item(),
        }


agent = DRQNAgent(CFG, net)
print(f"DRQNAgent ready. Replay: {CFG.replay_memory_size:,} slots")

# Cell 9 — Evaluate + Training loop


In [ ]:

def evaluate(agent, cfg, n_eval_episodes=None, eval_time=120):
    if n_eval_episodes is None:
        n_eval_episodes = cfg.n_eval_episodes

    agent.module.eval()

    eval_env = DoomEnv(cfg, action_builder)
    eval_map = cfg.map_ids_test[ep % len(cfg.map_ids_test)]
    eval_env.start(map_id=eval_map,
                   episode_time=eval_time,
                   log_events=False)
    
    total_kills = 0.0
    total_kd    = 0.0

    for ep in range(n_eval_episodes):
        agent.reset()   # FIX: fresh LSTM memory every episode
        last_states = []
        eval_map = cfg.map_ids_test[ep % len(cfg.map_ids_test)]
        if ep == 0:
            eval_env.start(map_id=eval_map, episode_time=eval_time, log_events=False)
        else:
            eval_env.new_episode()   # map rotation handled by the env

        eval_env.observe_state(last_states)
        ep_kills  = 0
        ep_deaths = 0

        while not eval_env.is_final():
            # Use epsilon_end throughout eval for mostly-greedy behaviour
            action_id, _, _ = agent.select_action(last_states,
                                                   n_iter=cfg.epsilon_steps)
            try:
                eval_env.make_action(action_id)
            except vzd.ViZDoomUnexpectedExitException:
                break

            # FIX: use kill_count not frag_count
            if eval_env.properties and eval_env.prev_properties:
                dk = (eval_env.properties.get("kill_count", 0) -
                      eval_env.prev_properties.get("kill_count", 0))
                if dk > 0:
                    ep_kills += int(dk)
                dd = (eval_env.properties.get("death_count", 0) -
                      eval_env.prev_properties.get("death_count", 0))
                if dd > 0:
                    ep_deaths += int(dd)

            eval_env.observe_state(last_states)

        total_kills += ep_kills
        total_kd    += ep_kills / max(1, ep_deaths)

        if ep < n_eval_episodes - 1:
            eval_env.new_episode()

    eval_env.close()
    agent.module.train()
    return total_kills / n_eval_episodes, total_kd / n_eval_episodes


# ─────────────────────────────────────────────────────────────────────
def run_training(cfg):
    env    = DoomEnv(cfg, action_builder)
    map_id = np.random.choice(cfg.map_ids_train)
    env.start(map_id=map_id, episode_time=cfg.train_episode_time, log_events=False)
    writer     = SummaryWriter("runs/drqn_doom")
    best_score = -1.0
    ep_count   = 0
    ep_reward, ep_kills, ep_deaths = 0.0, 0, 0
    loss_accum = {"loss_td": [], "loss_gf": [], "mean_q": []}

    # Initial observation before the loop
    last_states = []
    env.observe_state(last_states)
    agent.reset()

    print(f"Starting training for {cfg.total_steps:,} steps...")
    print(f"Action space: {cfg.n_actions} actions")
    print(f"Epsilon: {cfg.epsilon_start} → {cfg.epsilon_end} "
          f"over {cfg.epsilon_steps:,} steps")

    for n_iter in range(1, cfg.total_steps + 1):

        # last_states[-1] is s_t (set by observe_state at end of prev iter or init)
        action_id, pred_features, eps = agent.select_action(last_states, n_iter)

        # ── Take action ───────────────────────────────────────────────
        try:
            reward = env.make_action(action_id)
        except vzd.ViZDoomUnexpectedExitException:
            print(f"\n  [WARN] VizDoom crash at step {n_iter:,} — restarting")
            env.close()
            env    = DoomEnv(cfg, action_builder)
            map_id = np.random.choice(cfg.map_ids_train)
            env.start(map_id=map_id, episode_time=cfg.train_episode_time, log_events=False)
            agent.reset()
            last_states = []
            env.observe_state(last_states)
            ep_reward, ep_kills, ep_deaths = 0.0, 0, 0
            continue

        ep_reward += reward
        is_done    = env.is_final()

        if env.properties and env.prev_properties:
            dk = (env.properties.get("kill_count", 0) -
                  env.prev_properties.get("kill_count", 0))
            if dk > 0:
                ep_kills += int(dk)
            dd = (env.properties.get("death_count", 0) -
                  env.prev_properties.get("death_count", 0))
            if dd > 0:
                ep_deaths += int(dd)

        # Store (s_t, a_t, r_t, done_t) — last_states[-1] is still s_t here
        agent.store_transition(last_states, action_id, reward, is_done)

        # ── Train ─────────────────────────────────────────────────────
        losses = agent.train_step(n_iter)
        if losses:
            for k in loss_accum:
                if k in losses:
                    loss_accum[k].append(losses[k])

        # ── Episode end ───────────────────────────────────────────────
        if is_done:
            ep_count += 1
            kd = ep_kills / max(1, ep_deaths)
            writer.add_scalar("episode/reward",  ep_reward, n_iter)
            writer.add_scalar("episode/kills",   ep_kills,  n_iter)
            writer.add_scalar("episode/kd",      kd,        n_iter)
            writer.add_scalar("episode/epsilon", eps,       n_iter)

            if ep_count % 5 == 0:
                avg_td = np.mean(loss_accum["loss_td"]) if loss_accum["loss_td"] else 0.0
                avg_gf = np.mean(loss_accum["loss_gf"]) if loss_accum["loss_gf"] else 0.0
                avg_q  = np.mean(loss_accum["mean_q"])  if loss_accum["mean_q"]  else 0.0
                print(f"[{n_iter:>8,}/{cfg.total_steps:,}] "
                      f"ep={ep_count} rew={ep_reward:.1f} "
                      f"K={ep_kills} D={ep_deaths} K/D={kd:.2f} "
                      f"td={avg_td:.4f} gf={avg_gf:.4f} "
                      f"q={avg_q:.3f} eps={eps:.3f} "
                      f"mem={agent.replay_memory.size:,}")
                loss_accum = {"loss_td": [], "loss_gf": [], "mean_q": []}
                writer.flush()   # ← add this


            ep_reward, ep_kills, ep_deaths = 0.0, 0, 0
            env.new_episode()
            agent.reset()   # FIX: reset LSTM hidden state each episode
            last_states = []

        # ── Observe s_{t+1} for next iteration ───────────────────────
        # (If is_done, this observes the new episode's initial frame.)
        env.observe_state(last_states)

        # ── Periodic scalar logging ───────────────────────────────────
        if n_iter % (cfg.log_frequency * cfg.update_frequency) == 0:
            if loss_accum["loss_td"]:
                writer.add_scalar("train/loss_td",
                                  np.mean(loss_accum["loss_td"]), n_iter)
                writer.add_scalar("train/loss_gf",
                                  np.mean(loss_accum["loss_gf"]), n_iter)
                writer.add_scalar("train/mean_q",
                                  np.mean(loss_accum["mean_q"]),  n_iter)

        # ── Evaluation ────────────────────────────────────────────────
        if n_iter % cfg.eval_freq == 0:
            # Close training env during eval so VizDoom isn't double-loaded
            env.close()
            avg_kills, avg_kd = evaluate(agent, cfg, eval_time=cfg.eval_time)
            print(f"  [EVAL @ {n_iter:,}] avg_kills={avg_kills:.1f} "
                  f"avg_K/D={avg_kd:.2f}")
            writer.add_scalar("eval/avg_kills", avg_kills, n_iter)
            writer.add_scalar("eval/avg_kd",    avg_kd,    n_iter)
            if avg_kills > best_score:
                best_score = avg_kills
                torch.save(agent.module.state_dict(),
                           f"best_arnold_{n_iter}.pth")
                print(f"  New best! Saved best_arnold_{n_iter}.pth")
            agent.module.train()
            map_id = np.random.choice(cfg.map_ids_train)
            env    = DoomEnv(cfg, action_builder)
            env.start(map_id=map_id, episode_time=cfg.train_episode_time, log_events=False)
            agent.reset()
            last_states = []
            env.observe_state(last_states)  # observe initial frame of new training env
            loss_accum = {"loss_td": [], "loss_gf": [], "mean_q": []}

        # ── Periodic checkpoint ───────────────────────────────────────
        if cfg.dump_freq > 0 and n_iter % cfg.dump_freq == 0:
            torch.save(agent.module.state_dict(), f"checkpoint_{n_iter}.pth")

    env.close()
    torch.save(agent.module.state_dict(), "final_model.pth")
    print(f"Final model saved. Steps: {cfg.total_steps:,}")
    writer.close()


print("Training function ready")

# Cell 10 — Sanity check


In [ ]:

print("Running sanity checks...")

# Action space
assert len(action_builder.doom_actions) == CFG.n_actions
print(f"  Action space: {CFG.n_actions} actions  ✓")

# Network forward pass — use full seq_len so shapes match training
B   = 2
T   = CFG.hist_size + CFG.n_rec_updates   # same as seq_len in train_step
dummy_screens = torch.randn(B, T, CFG.n_fm, CFG.height, CFG.width).to(DEVICE) * 255
dummy_vars    = [torch.randint(0, 100, (B, T)).to(DEVICE)
                 for _ in range(CFG.n_variables)]
h0 = torch.zeros(CFG.n_rec_layers, B, CFG.hidden_dim).to(DEVICE)
c0 = torch.zeros(CFG.n_rec_layers, B, CFG.hidden_dim).to(DEVICE)

net.eval()
with torch.no_grad():
    out_sc, out_gf, _ = net(dummy_screens, dummy_vars, (h0, c0))
assert out_sc.shape == (B, T, CFG.n_actions),   f"Q shape wrong: {out_sc.shape}"
if out_gf is not None:
    assert out_gf.shape == (B, T, CFG.n_features), f"GF shape wrong: {out_gf.shape}"
net.train()
print(f"  Q-values shape  : {out_sc.shape}  ✓")
print(f"  Features shape  : {out_gf.shape if out_gf is not None else None}  ✓")

# Single-step inference shape (T=1, persistent hidden state)
net.eval()
s1 = torch.randn(1, 1, CFG.n_fm, CFG.height, CFG.width).to(DEVICE) * 255
v1 = [torch.zeros(1, 1, dtype=torch.long).to(DEVICE) for _ in range(CFG.n_variables)]
h1 = torch.zeros(CFG.n_rec_layers, 1, CFG.hidden_dim).to(DEVICE)
c1 = torch.zeros(CFG.n_rec_layers, 1, CFG.hidden_dim).to(DEVICE)
with torch.no_grad():
    q1, _, _ = net(s1, v1, (h1, c1))
assert q1.shape == (1, 1, CFG.n_actions)
net.train()
print(f"  Single-step inference shape: {q1.shape}  ✓")

# Replay memory — fill past wrap point, check batch shapes
hist_size_train = CFG.hist_size + CFG.n_rec_updates - 1
mem = ReplayMemory(200,
                   (CFG.n_fm, CFG.height, CFG.width),
                   CFG.n_variables,
                   CFG.n_features)
for i in range(150):   # deliberately past the 200-slot wrap
    sc = np.random.randint(0, 255, (CFG.n_fm, CFG.height, CFG.width), dtype=np.uint8)
    
    mem.add(sc, [50], [1, 0], i % CFG.n_actions, 0.5, i % 20 == 19)
batch = mem.get_batch(4, hist_size_train)
seq_len = hist_size_train + 1
assert batch["screens"].shape == (4, seq_len, CFG.n_fm, CFG.height, CFG.width)
assert batch["actions"].shape  == (4, hist_size_train)
print(f"  Replay batch screens: {batch['screens'].shape}  ✓")
print(f"  Replay batch actions: {batch['actions'].shape}  ✓")

# DoomEnv round-trip
print("  Testing DoomEnv round-trip...")
test_env = DoomEnv(CFG, action_builder)
test_env.start(map_id=1, episode_time=10, log_events=False)
test_states = []
test_env.observe_state(test_states)
assert len(test_states) == CFG.hist_size
for _ in range(5):
    if test_env.is_final():
        break
    test_env.make_action(0)
    test_env.observe_state(test_states)
    assert len(test_states) == CFG.hist_size
test_env.close()
print("  DoomEnv round-trip  ✓")

# Config value checks
assert CFG.reward_values["DISTANCE"] > 0,    "DISTANCE reward must be > 0"
assert CFG.game_variables[0][1] >= 201,      "health bucket must cover 200"
assert CFG.n_rec_updates >= 4,               "n_rec_updates too low"
assert CFG.n_eval_episodes >= 10,            "n_eval_episodes too low"
print(f"  Config sanity  ✓")

print()
print("All sanity checks passed!")

# Cell 10 — Run training


In [ ]:

import traceback

try:
    import psutil
    m = psutil.virtual_memory()
    print(f"RAM: {m.total/1e9:.1f} GB total, {m.available/1e9:.1f} GB free")
except ImportError:
    pass

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name} — {props.total_memory/1e9:.1f} GB")
print()

try:
    run_training(CFG)
except Exception:
    print()
    print("=" * 60)
    print("TRAINING FAILED:")
    print("=" * 60)
    traceback.print_exc()